# Twitter / X Sentiment Investing Strategy

## Strategy Overview
This notebook builds a **full end-to-end quantitative trading strategy** driven by Twitter/X social sentiment. The pipeline:

1. **Simulate / Load Twitter sentiment data** — tweet counts, sentiment scores, engagement, and buzz metrics per ticker per day.
2. **Engineer sentiment features** — rolling sentiment momentum, sentiment divergence from price, engagement ratio, bullish/bearish ratio.
3. **Generate monthly signals** — rank stocks cross-sectionally on composite sentiment score.
4. **Portfolio construction** — top-N long + optional short leg, equally weighted or volatility-scaled.
5. **Backtest with transaction costs** — realistic slippage + commission model.
6. **Performance analytics** — Sharpe ratio, max drawdown, alpha vs. NASDAQ (QQQ) and S&P 500 (SPY).
7. **Sentiment-Price Divergence Signal** — ML-enhanced signal identifying when sentiment leads price.

---
### Packages Required
```
pip install pandas numpy matplotlib yfinance scikit-learn vaderSentiment textblob pypfopt
```

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import datetime as dt
import warnings
import yfinance as yf
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
np.random.seed(42)

# ── Strategy Config ──────────────────────────────────────────────────
START_DATE          = '2019-01-01'
END_DATE            = '2023-12-31'
TOP_N_STOCKS        = 10        # long leg size
BOTTOM_N_STOCKS     = 10        # short leg size (set 0 for long-only)
TRANSACTION_COST_BP = 10        # one-way cost in basis points
BENCHMARK_TICKER    = 'QQQ'     # benchmark to compare against
REBALANCE_FREQ      = 'M'       # monthly rebalancing

# Universe — large-cap tech + growth stocks (typical high-Twitter-activity names)
UNIVERSE = [
    'AAPL','MSFT','GOOGL','AMZN','META','TSLA','NVDA','AMD','NFLX','SNAP',
    'TWTR','UBER','LYFT','PINS','ROKU','SHOP','SQ','PYPL','CRM','ORCL',
    'INTC','QCOM','MU','BIDU','JD','BABA','SPOT','ZM','DOCU','PLTR',
    'RBLX','COIN','HOOD','SOFI','AFRM','DKNG','PENN','MARA','RIOT','GME'
]

print(f'Universe size: {len(UNIVERSE)} stocks')
print(f'Backtest window: {START_DATE} → {END_DATE}')

## 1. Simulate Realistic Twitter Sentiment Data

Real Twitter/X data can be sourced from:
- **StockTwits API** (free, financial-specific)
- **RapidAPI Twitter scrapers**
- **Quiverquant** (paid, clean financial Twitter data)
- **Kaggle datasets** — search *"Twitter stock sentiment"*
- **Refinitiv / Bloomberg** sentiment feeds (enterprise)

Below we simulate a realistic dataset using a regime-switching model that:
- Injects **sentiment momentum** (sentiment tends to persist)
- Models **volume spikes** around earnings and news events
- Adds **mean-reverting noise** so the signal isn't perfect
- Correlates sentiment loosely with future 1-month returns (the alpha we want to capture)

In [ ]:
def simulate_sentiment_data(tickers, start, end, seed=42):
    """
    Simulate daily Twitter sentiment data per ticker.
    
    Columns produced:
      tweet_count      — total tweets mentioning the stock that day
      positive_count   — tweets with positive sentiment
      negative_count   — tweets with negative sentiment
      neutral_count    — neutral tweets
      avg_sentiment    — mean VADER compound score [-1, 1]
      bullish_ratio    — positive / (positive + negative)
      engagement_ratio — (likes + retweets) per tweet (proxy)
      influencer_score — weighted sentiment from high-follower accounts
    """
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, end)   # business days only
    records = []

    for ticker in tickers:
        # Ticker-specific base popularity (larger names → more tweets)
        base_vol = rng.integers(200, 5000)
        
        # Latent sentiment process: AR(1) mean-reverting
        phi        = 0.82          # persistence
        sigma_sent = 0.15
        sentiment  = np.zeros(len(dates))
        sentiment[0] = rng.normal(0, sigma_sent)
        for t in range(1, len(dates)):
            sentiment[t] = phi * sentiment[t-1] + rng.normal(0, sigma_sent * np.sqrt(1 - phi**2))
        sentiment = np.clip(sentiment, -1, 1)

        # Tweet volume: log-normal with random spikes (earnings, news)
        log_vol   = np.log(base_vol) + rng.normal(0, 0.3, len(dates))
        spike_idx = rng.choice(len(dates), size=int(len(dates) * 0.05), replace=False)
        log_vol[spike_idx] += rng.uniform(1.0, 2.5, len(spike_idx))
        tweet_count = np.round(np.exp(log_vol)).astype(int)

        # Derive positive / negative splits from sentiment
        pos_frac = 0.5 + 0.4 * sentiment + rng.normal(0, 0.05, len(dates))
        pos_frac = np.clip(pos_frac, 0.05, 0.95)
        neg_frac = (1 - pos_frac) * rng.uniform(0.3, 0.8, len(dates))
        neg_frac = np.clip(neg_frac, 0.02, 1 - pos_frac)
        neu_frac = np.clip(1 - pos_frac - neg_frac, 0, 1)

        positive_count = np.round(tweet_count * pos_frac).astype(int)
        negative_count = np.round(tweet_count * neg_frac).astype(int)
        neutral_count  = tweet_count - positive_count - negative_count
        neutral_count  = np.clip(neutral_count, 0, tweet_count)

        # Bullish ratio — smoother version of raw sentiment
        safe_denom    = positive_count + negative_count
        bullish_ratio = np.where(safe_denom > 0, positive_count / safe_denom, 0.5)

        # Engagement: high-sentiment tweets get amplified (likes/RT)
        engagement_ratio  = 5 + 20 * np.abs(sentiment) + rng.exponential(3, len(dates))

        # Influencer score: sparse high-impact tweets
        influencer_score = sentiment * rng.beta(2, 5, len(dates)) * tweet_count * 0.002

        for i, date in enumerate(dates):
            records.append({
                'date'             : date,
                'symbol'           : ticker,
                'tweet_count'      : tweet_count[i],
                'positive_count'   : positive_count[i],
                'negative_count'   : negative_count[i],
                'neutral_count'    : max(0, neutral_count[i]),
                'avg_sentiment'    : round(sentiment[i], 4),
                'bullish_ratio'    : round(bullish_ratio[i], 4),
                'engagement_ratio' : round(engagement_ratio[i], 2),
                'influencer_score' : round(influencer_score[i], 4),
            })

    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index(['date', 'symbol']).sort_index()
    return df


sentiment_df = simulate_sentiment_data(UNIVERSE, START_DATE, END_DATE)
print(f'Sentiment data shape: {sentiment_df.shape}')
print(f'Date range: {sentiment_df.index.get_level_values(0).min().date()} → {sentiment_df.index.get_level_values(0).max().date()}')
sentiment_df.head(10)

## 2. Exploratory Data Analysis

Before building signals, understand the data distributions and cross-sectional variation.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Tweet volume distribution (log scale)
ax = axes[0, 0]
sentiment_df['tweet_count'].apply(np.log1p).hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Log Tweet Volume Distribution')
ax.set_xlabel('log(tweet_count + 1)')
ax.set_ylabel('Frequency')

# 2. Avg sentiment distribution
ax = axes[0, 1]
sentiment_df['avg_sentiment'].hist(bins=60, ax=ax, color='mediumseagreen', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Neutral')
ax.set_title('Avg Sentiment Distribution')
ax.set_xlabel('Sentiment Score [-1, 1]')
ax.legend()

# 3. Bullish ratio distribution
ax = axes[0, 2]
sentiment_df['bullish_ratio'].hist(bins=60, ax=ax, color='coral', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='50/50')
ax.set_title('Bullish Ratio Distribution')
ax.set_xlabel('Bullish Ratio [0, 1]')
ax.legend()

# 4. Per-ticker average tweet count (cross-section)
ax = axes[1, 0]
per_ticker = sentiment_df.groupby('symbol')['tweet_count'].mean().sort_values(ascending=False)
per_ticker.head(20).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 20 Tickers by Avg Daily Tweet Volume')
ax.set_xlabel('Ticker')
ax.set_ylabel('Avg Tweets / Day')
ax.tick_params(axis='x', rotation=45)

# 5. Sentiment time series for TSLA
ax = axes[1, 1]
tsla_sent = sentiment_df.xs('TSLA', level=1)['avg_sentiment'].resample('W').mean()
tsla_sent.plot(ax=ax, color='gold', linewidth=1.2)
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.fill_between(tsla_sent.index, tsla_sent, 0,
                where=(tsla_sent > 0), alpha=0.3, color='green', label='Bullish')
ax.fill_between(tsla_sent.index, tsla_sent, 0,
                where=(tsla_sent < 0), alpha=0.3, color='red', label='Bearish')
ax.set_title('TSLA Weekly Avg Sentiment')
ax.set_ylabel('Sentiment Score')
ax.legend()

# 6. Engagement ratio vs sentiment (scatter)
ax = axes[1, 2]
sample = sentiment_df.sample(5000, random_state=42)
ax.scatter(sample['avg_sentiment'], np.log1p(sample['engagement_ratio']),
           alpha=0.15, s=8, color='purple')
ax.set_title('Sentiment vs Log Engagement')
ax.set_xlabel('Avg Sentiment')
ax.set_ylabel('log(Engagement Ratio)')

plt.suptitle('Twitter Sentiment Dataset — EDA', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering — Composite Sentiment Score

Raw daily sentiment is noisy. We engineer **multi-horizon rolling features** to capture:
- **Short-term buzz** (3-day rolling)
- **Medium-term trend** (10-day rolling)
- **Sentiment momentum** — change in sentiment over time
- **Sentiment surprise** — deviation from trailing average (z-score)
- **Volume-weighted sentiment** — weight sentiment by engagement
- **Bull/Bear divergence** — gap between bullish ratio and 0.5 baseline

In [ ]:
def engineer_sentiment_features(df):
    """
    Compute multi-horizon rolling sentiment features per symbol.
    All features are computed within-symbol to prevent lookahead bias.
    """
    result = []

    for symbol, grp in df.groupby(level='symbol'):
        g = grp.copy().droplevel('symbol')  # daily index

        # ── Rolling sentiment means ───────────────────────────────────────
        g['sent_3d']  = g['avg_sentiment'].rolling(3,  min_periods=2).mean()
        g['sent_10d'] = g['avg_sentiment'].rolling(10, min_periods=5).mean()
        g['sent_21d'] = g['avg_sentiment'].rolling(21, min_periods=10).mean()

        # ── Sentiment momentum (short vs. medium term) ────────────────────
        g['sent_momentum_3_10']  = g['sent_3d']  - g['sent_10d']
        g['sent_momentum_10_21'] = g['sent_10d'] - g['sent_21d']

        # ── Sentiment z-score (standardised surprise) ─────────────────────
        rolling_std = g['avg_sentiment'].rolling(21, min_periods=10).std()
        g['sent_zscore'] = (g['avg_sentiment'] - g['sent_21d']) / (rolling_std + 1e-8)

        # ── Volume-weighted sentiment ─────────────────────────────────────
        g['vol_weighted_sent'] = (
            (g['avg_sentiment'] * g['tweet_count']).rolling(10, min_periods=5).sum()
            / g['tweet_count'].rolling(10, min_periods=5).sum()
        )

        # ── Tweet volume features ──────────────────────────────────────────
        g['log_tweet_vol']    = np.log1p(g['tweet_count'])
        g['tweet_vol_ratio']  = g['tweet_count'] / (g['tweet_count'].rolling(21, min_periods=10).mean() + 1)

        # ── Bull/Bear divergence ───────────────────────────────────────────
        g['bull_bear_div']     = g['bullish_ratio'] - 0.5
        g['bull_bear_div_3d']  = g['bull_bear_div'].rolling(3,  min_periods=2).mean()
        g['bull_bear_div_10d'] = g['bull_bear_div'].rolling(10, min_periods=5).mean()

        # ── Engagement-weighted score ──────────────────────────────────────
        g['engagement_sent'] = (
            (g['avg_sentiment'] * g['engagement_ratio']).rolling(10, min_periods=5).sum()
            / (g['engagement_ratio'].rolling(10, min_periods=5).sum() + 1e-8)
        )

        # ── Influencer signal (sparse, high-value) ─────────────────────────
        g['influencer_signal'] = g['influencer_score'].rolling(5, min_periods=2).mean()

        g['symbol'] = symbol
        result.append(g)

    out = pd.concat(result).reset_index().set_index(['date', 'symbol'])
    return out


features_df = engineer_sentiment_features(sentiment_df)
print(f'Feature matrix shape: {features_df.shape}')
print(f'Columns: {features_df.columns.tolist()}')
features_df.head()

## 4. Aggregate to Monthly Level & Build Composite Signal

We resample to month-end to form a **monthly rebalancing signal**. The composite score is a weighted combination of our engineered features.

**Composite Score** = w₁ × sentiment_momentum + w₂ × vol_weighted_sent + w₃ × bull_bear_div + w₄ × sent_zscore + w₅ × engagement_sent + w₆ × influencer_signal

We then **cross-sectionally rank** stocks each month — this is a key quant technique that:
- Removes market-wide sentiment biases
- Focuses on *relative* not *absolute* sentiment
- Reduces exposure to systematic sentiment swings

In [ ]:
# Feature weights (can be optimised; here set by financial intuition)
WEIGHTS = {
    'sent_momentum_3_10' : 0.20,
    'vol_weighted_sent'  : 0.25,
    'bull_bear_div_10d'  : 0.20,
    'sent_zscore'        : 0.15,
    'engagement_sent'    : 0.15,
    'influencer_signal'  : 0.05,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1'


def build_monthly_signal(features_df, weights):
    """
    Resample to month-end frequency, compute composite score,
    and cross-sectionally rank each month.
    """
    feature_cols = list(weights.keys())

    # Resample: take last observation of month (or mean for smoother signal)
    monthly = (
        features_df[feature_cols]
        .reset_index('symbol')
        .groupby([pd.Grouper(freq='M'), 'symbol'])
        .last()
    )

    # Cross-sectional z-score normalisation (removes month-level mean)
    def xs_normalise(df):
        return (df - df.mean()) / (df.std() + 1e-8)

    normalised = monthly.groupby(level='date')[feature_cols].transform(xs_normalise)

    # Composite score
    composite = sum(normalised[col] * w for col, w in weights.items())
    monthly['composite_score'] = composite

    # Cross-sectional rank (1 = highest composite score)
    monthly['rank'] = (
        monthly.groupby(level='date')['composite_score']
        .transform(lambda x: x.rank(ascending=False))
    )

    return monthly


monthly_signal = build_monthly_signal(features_df, WEIGHTS)
print(f'Monthly signal shape: {monthly_signal.shape}')
monthly_signal[['composite_score', 'rank']].head(20)

### Visualise Cross-Sectional Composite Score Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of composite scores across all months
monthly_signal['composite_score'].hist(bins=60, ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('Distribution of Monthly Composite Sentiment Scores')
ax1.set_xlabel('Composite Score (normalised)')
ax1.set_ylabel('Count')

# Heatmap: per-ticker average rank over time (lower rank = consistently high sentiment)
pivot = (
    monthly_signal['rank']
    .unstack('symbol')
    .resample('6M').mean()
)
avg_rank = pivot.mean().sort_values()
avg_rank.plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Average Monthly Rank per Stock\n(lower = more consistently bullish sentiment)')
ax2.set_xlabel('Ticker')
ax2.set_ylabel('Average Rank')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Download Price Data for Universe

In [ ]:
print('Downloading price data...')
prices_raw = yf.download(
    tickers=UNIVERSE + [BENCHMARK_TICKER, 'SPY'],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

prices = prices_raw['Close'].copy()
print(f'Price data shape: {prices.shape}')

# Drop tickers with too many missing values
coverage = prices.notna().mean()
valid_tickers = coverage[coverage > 0.7].index.tolist()
prices = prices[valid_tickers]
print(f'Valid tickers after coverage filter: {len(valid_tickers)}')

prices.tail()

## 6. Compute Forward Returns (Ground Truth)

For each month-end rebalance date, compute the **forward 1-month return** for every stock — this is what our signal is trying to predict. We also perform an **IC (Information Coefficient) analysis** to measure raw signal quality before backtesting.

In [ ]:
# Monthly log returns for each stock
log_returns_daily = np.log(prices[UNIVERSE]).diff()

monthly_returns = np.exp(
    log_returns_daily.resample('M').sum()
).sub(1)  # arithmetic monthly return

# Align signal with NEXT month's return (shift returns back by 1 month)
fwd_returns = monthly_returns.shift(-1)   # next month return
fwd_returns.index.name = 'date'

# Melt to long format for merging
fwd_long = fwd_returns.stack().rename('fwd_return_1m').rename_axis(['date', 'symbol'])

# Merge signal with forward returns
signal_with_returns = monthly_signal[['composite_score', 'rank']].join(fwd_long, how='inner')
signal_with_returns = signal_with_returns.dropna()

print(f'Signal + returns merged: {signal_with_returns.shape}')
signal_with_returns.head(10)

## 7. Information Coefficient (IC) Analysis

The **IC** measures the rank correlation between our signal and forward returns each month.
- **IC > 0.05** → signal has genuine predictive power
- **ICIR > 0.5** → signal is stable across time (Information Coefficient / Std Dev)

This is how professional quants evaluate signals *before* building portfolios.

In [ ]:
def compute_monthly_ic(signal_returns_df, signal_col='composite_score', return_col='fwd_return_1m'):
    """Compute monthly Spearman rank IC between signal and forward returns."""
    ic_series = {}
    for date, group in signal_returns_df.groupby(level='date'):
        if len(group) < 5:
            continue
        rho, _ = stats.spearmanr(group[signal_col], group[return_col])
        ic_series[date] = rho
    return pd.Series(ic_series, name='IC')


ic_series = compute_monthly_ic(signal_with_returns)

ic_mean  = ic_series.mean()
ic_std   = ic_series.std()
icir     = ic_mean / ic_std
ic_pos   = (ic_series > 0).mean()

print('═' * 45)
print(f'  Mean IC         : {ic_mean:>8.4f}')
print(f'  IC Std Dev      : {ic_std:>8.4f}')
print(f'  IC IR           : {icir:>8.4f}')
print(f'  % Months IC > 0 : {ic_pos:>7.1%}')
print('═' * 45)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# IC over time
ax = axes[0]
ic_series.plot(ax=ax, color='steelblue', linewidth=1.3)
ic_series.rolling(6).mean().plot(ax=ax, color='orange', linewidth=2.2, label='6M rolling mean')
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.fill_between(ic_series.index, ic_series, 0,
                where=(ic_series > 0), alpha=0.2, color='green')
ax.fill_between(ic_series.index, ic_series, 0,
                where=(ic_series < 0), alpha=0.2, color='red')
ax.set_title(f'Monthly IC  |  Mean={ic_mean:.3f}  ICIR={icir:.2f}')
ax.set_ylabel('Spearman IC')
ax.legend()

# IC distribution
ax = axes[1]
ic_series.hist(bins=25, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.axvline(ic_mean, color='orange', linestyle='--', linewidth=1.5, label=f'Mean = {ic_mean:.3f}')
ax.set_title('IC Distribution')
ax.set_xlabel('IC')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Quintile Return Analysis

A quintile analysis separates stocks into 5 buckets by composite score each month and measures average forward returns per bucket. A **monotonically increasing return from Q1 to Q5** is the hallmark of a genuine alpha signal.

In [ ]:
def quintile_analysis(df, signal_col='composite_score', return_col='fwd_return_1m', n_buckets=5):
    """Cross-sectional quintile bucketing within each month."""
    df = df.copy()
    def assign_bucket(g):
        g['bucket'] = pd.qcut(g[signal_col], q=n_buckets, labels=False, duplicates='drop')
        return g
    df = df.groupby(level='date', group_keys=False).apply(assign_bucket)
    quintile_returns = df.groupby('bucket')[return_col].mean()
    quintile_returns.index = [f'Q{int(i)+1}' for i in quintile_returns.index]
    return quintile_returns, df


quintile_rets, df_with_buckets = quintile_analysis(signal_with_returns)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Quintile bar chart
ax = axes[0]
colors = ['#d9534f' if r < 0 else '#5cb85c' for r in quintile_rets]
quintile_rets.mul(100).plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.6)
ax.set_title('Average Forward 1M Return by Sentiment Quintile\n(Q1 = Lowest Sentiment, Q5 = Highest)')
ax.set_ylabel('Avg Monthly Return (%)')
ax.set_xlabel('Sentiment Quintile')
ax.tick_params(axis='x', rotation=0)

# Long–Short spread over time
ax = axes[1]
bucket_ts = df_with_buckets.groupby(['date', 'bucket'])['fwd_return_1m'].mean().unstack()
long_short_spread = bucket_ts.get(4, 0) - bucket_ts.get(0, 0)   # Q5 - Q1
long_short_spread.cumsum().plot(ax=ax, color='purple', linewidth=1.8)
ax.axhline(0, color='red', linestyle='--')
ax.set_title('Cumulative Long-Short Spread (Q5 − Q1)')
ax.set_ylabel('Cumulative Return')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))

plt.tight_layout()
plt.show()

print('\nQuintile Average Monthly Returns:')
print((quintile_rets * 100).round(3).to_string())
print(f'\nLong-Short Spread (Q5 - Q1): {(quintile_rets.iloc[-1] - quintile_rets.iloc[0]) * 100:.2f}%/month')

## 9. Portfolio Construction with Monthly Rebalancing

We build two portfolios:
- **Long-Only**: Equal weight top-N stocks by composite score each month.
- **Long-Short**: Long top-N, Short bottom-N.

We apply **realistic transaction costs**: 10bps one-way (20bps round-trip) to penalise high turnover. Portfolio returns include slippage on positions that change each month.

In [ ]:
def build_portfolio(monthly_signal, prices, top_n=TOP_N_STOCKS, bottom_n=BOTTOM_N_STOCKS,
                    cost_bps=TRANSACTION_COST_BP):
    """
    Build daily portfolio returns with monthly rebalancing.
    Returns dict of portfolio return series.
    """
    daily_log_returns = np.log(prices).diff()
    universe_tickers  = [t for t in UNIVERSE if t in prices.columns]

    rebalance_dates = monthly_signal.index.get_level_values('date').unique().sort_values()

    long_only_returns = pd.Series(dtype=float)
    long_short_returns = pd.Series(dtype=float)

    prev_long_wts  = {}
    prev_short_wts = {}

    for i, reb_date in enumerate(rebalance_dates[:-1]):
        next_reb  = rebalance_dates[i + 1]

        # Signal as of reb_date (available end of month, trade next day open)
        try:
            snapshot = monthly_signal.xs(reb_date, level='date')[['composite_score']].dropna()
        except KeyError:
            continue

        # Only keep tickers we have prices for
        snapshot = snapshot[snapshot.index.isin(universe_tickers)]
        if len(snapshot) < top_n:
            continue

        snapshot_sorted = snapshot['composite_score'].sort_values(ascending=False)
        long_tickers  = snapshot_sorted.head(top_n).index.tolist()
        short_tickers = snapshot_sorted.tail(bottom_n).index.tolist() if bottom_n > 0 else []

        long_wts  = {t: 1.0 / top_n   for t in long_tickers}
        short_wts = {t: 1.0 / bottom_n for t in short_tickers} if short_tickers else {}

        # Transaction cost: turnover × cost_bps / 10000
        def turnover_cost(new_wts, old_wts):
            all_tickers = set(new_wts) | set(old_wts)
            tv = sum(abs(new_wts.get(t, 0) - old_wts.get(t, 0)) for t in all_tickers)
            return tv * cost_bps / 10000

        long_cost  = turnover_cost(long_wts,  prev_long_wts)
        short_cost = turnover_cost(short_wts, prev_short_wts)

        # Daily returns for holding period
        period_slice = daily_log_returns.loc[reb_date:next_reb]

        if len(period_slice) == 0:
            continue

        # Long leg
        long_cols = [t for t in long_tickers if t in period_slice.columns]
        if long_cols:
            daily_long = period_slice[long_cols].mean(axis=1)
            daily_long.iloc[0] -= long_cost   # deduct TC on first day
            long_only_returns = pd.concat([long_only_returns, daily_long])

        # Long-Short leg
        if short_tickers:
            short_cols = [t for t in short_tickers if t in period_slice.columns]
            if short_cols:
                daily_short    = period_slice[short_cols].mean(axis=1)
                daily_ls       = (daily_long - daily_short) / 2
                daily_ls.iloc[0] -= (long_cost + short_cost) / 2
                long_short_returns = pd.concat([long_short_returns, daily_ls])

        prev_long_wts  = long_wts
        prev_short_wts = short_wts

    return {
        'long_only'   : long_only_returns.sort_index(),
        'long_short'  : long_short_returns.sort_index() if len(long_short_returns) > 0 else pd.Series(dtype=float),
    }


portfolios = build_portfolio(monthly_signal, prices)
print(f'Long-Only return series length  : {len(portfolios["long_only"])}')
print(f'Long-Short return series length : {len(portfolios["long_short"])}')

## 10. Performance Analytics

We compute a full suite of risk-adjusted performance metrics and compare to QQQ and SPY.

In [ ]:
def performance_stats(returns_series, name='Portfolio', rf_daily=0.0):
    """Compute annualised performance statistics from daily log returns."""
    r = returns_series.dropna()
    if len(r) == 0:
        return {}

    cum_ret       = np.exp(r.cumsum()) - 1
    total_return  = cum_ret.iloc[-1]
    n_years       = len(r) / 252
    ann_return    = (1 + total_return) ** (1 / n_years) - 1
    ann_vol       = r.std() * np.sqrt(252)
    sharpe        = (ann_return - rf_daily * 252) / ann_vol if ann_vol > 0 else np.nan

    roll_max      = cum_ret.cummax()
    drawdown      = cum_ret - roll_max
    max_dd        = drawdown.min()

    calmar        = ann_return / abs(max_dd) if max_dd != 0 else np.nan
    sortino_denom = r[r < 0].std() * np.sqrt(252)
    sortino       = ann_return / sortino_denom if sortino_denom > 0 else np.nan

    win_rate      = (r > 0).mean()

    return {
        'Name'             : name,
        'Total Return'     : f'{total_return:.1%}',
        'Ann. Return'      : f'{ann_return:.2%}',
        'Ann. Volatility'  : f'{ann_vol:.2%}',
        'Sharpe Ratio'     : f'{sharpe:.2f}',
        'Sortino Ratio'    : f'{sortino:.2f}',
        'Calmar Ratio'     : f'{calmar:.2f}',
        'Max Drawdown'     : f'{max_dd:.1%}',
        'Daily Win Rate'   : f'{win_rate:.1%}',
    }


# Benchmark returns
qqq_ret = np.log(prices[BENCHMARK_TICKER]).diff().dropna()
spy_ret = np.log(prices['SPY']).diff().dropna()

# Align all series
common_idx = portfolios['long_only'].index.intersection(qqq_ret.index).intersection(spy_ret.index)

lo_aligned = portfolios['long_only'].reindex(common_idx)
qqq_aligned = qqq_ret.reindex(common_idx)
spy_aligned = spy_ret.reindex(common_idx)

stats_rows = [
    performance_stats(lo_aligned, name='Sentiment Long-Only'),
    performance_stats(qqq_aligned, name=f'{BENCHMARK_TICKER} (Benchmark)'),
    performance_stats(spy_aligned, name='SPY'),
]

if len(portfolios['long_short']) > 100:
    ls_aligned = portfolios['long_short'].reindex(common_idx)
    stats_rows.insert(1, performance_stats(ls_aligned, name='Sentiment Long-Short'))

stats_df = pd.DataFrame(stats_rows).set_index('Name')
stats_df

## 11. Cumulative Returns Chart

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# ── Cumulative returns ───────────────────────────────────────────────
ax = axes[0]
cum_lo  = np.exp(lo_aligned.cumsum()) - 1
cum_qqq = np.exp(qqq_aligned.cumsum()) - 1
cum_spy = np.exp(spy_aligned.cumsum()) - 1

cum_lo.plot(ax=ax,  color='steelblue',     linewidth=2.0, label='Sentiment Long-Only')
cum_qqq.plot(ax=ax, color='darkorange',    linewidth=1.5, label=f'{BENCHMARK_TICKER} Benchmark', linestyle='--')
cum_spy.plot(ax=ax, color='gray',          linewidth=1.5, label='SPY',                           linestyle=':')

if len(portfolios['long_short']) > 100:
    cum_ls = np.exp(ls_aligned.cumsum()) - 1
    cum_ls.plot(ax=ax, color='mediumseagreen', linewidth=2.0, label='Sentiment Long-Short')

ax.set_title('Twitter Sentiment Strategy — Cumulative Returns', fontsize=14, fontweight='bold')
ax.set_ylabel('Cumulative Return')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.legend(fontsize=11)
ax.axhline(0, color='black', linewidth=0.8)

# ── Drawdown chart ───────────────────────────────────────────────────
ax = axes[1]
def drawdown_series(cum_ret):
    roll_max = cum_ret.cummax()
    return (cum_ret - roll_max) / (1 + roll_max)

dd_lo  = drawdown_series(cum_lo + 1)
dd_qqq = drawdown_series(cum_qqq + 1)

dd_lo.plot(ax=ax,  color='steelblue',  linewidth=1.5, label='Sentiment Long-Only')
dd_qqq.plot(ax=ax, color='darkorange', linewidth=1.5, label=f'{BENCHMARK_TICKER}', linestyle='--')
ax.fill_between(dd_lo.index, dd_lo, 0, alpha=0.25, color='steelblue')
ax.set_title('Drawdown Comparison', fontsize=13)
ax.set_ylabel('Drawdown')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.legend()

plt.tight_layout()
plt.show()

## 12. Rolling Sharpe Ratio & Alpha Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Rolling 12M Sharpe
ax = axes[0]
window = 252
rolling_sharpe_lo  = (lo_aligned.rolling(window).mean() * 252) / (lo_aligned.rolling(window).std() * np.sqrt(252))
rolling_sharpe_qqq = (qqq_aligned.rolling(window).mean() * 252) / (qqq_aligned.rolling(window).std() * np.sqrt(252))

rolling_sharpe_lo.plot(ax=ax,  color='steelblue',  linewidth=1.8, label='Sentiment Long-Only')
rolling_sharpe_qqq.plot(ax=ax, color='darkorange',  linewidth=1.8, label=f'{BENCHMARK_TICKER}', linestyle='--')
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.axhline(1, color='green', linestyle=':', linewidth=1, label='Sharpe = 1')
ax.set_title('Rolling 12-Month Sharpe Ratio')
ax.set_ylabel('Sharpe Ratio')
ax.legend()

# Monthly alpha (excess return vs benchmark)
ax = axes[1]
monthly_lo  = lo_aligned.resample('M').sum()
monthly_qqq = qqq_aligned.resample('M').sum()
alpha_monthly = monthly_lo - monthly_qqq

colors = ['green' if a > 0 else 'red' for a in alpha_monthly]
alpha_monthly.plot(kind='bar', ax=ax, color=colors, width=0.9, edgecolor='none')
ax.axhline(0, color='black', linewidth=1)
ax.set_title(f'Monthly Alpha vs {BENCHMARK_TICKER}')
ax.set_ylabel('Excess Return (log)')

# Thin x-axis labels
tick_labels = [label.get_text() if i % 6 == 0 else '' for i, label in enumerate(ax.get_xticklabels())]
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f'Annualised Alpha vs {BENCHMARK_TICKER}: {(alpha_monthly.mean() * 12):.2%}')
print(f'% Months Outperforming            : {(alpha_monthly > 0).mean():.1%}')

## 13. ML-Enhanced Signal: Sentiment-Price Divergence

### Hypothesis
When **sentiment is rising but price hasn't moved yet**, there is a potential *lead-lag* opportunity. We define a **Sentiment-Price Divergence (SPD)** feature and use Ridge Regression to predict next-month returns, combining sentiment features with price-based features.

This is an extension that makes the strategy more robust by blending sentiment alpha with technical signals.

In [ ]:
def build_ml_features(features_df, prices, universe):
    """
    Combine sentiment + price-based features for ML model.
    """
    # Monthly sentiment features
    sent_cols = ['sent_momentum_3_10', 'vol_weighted_sent', 'bull_bear_div_10d',
                 'sent_zscore', 'engagement_sent', 'influencer_signal']

    monthly_sent = (
        features_df[sent_cols]
        .reset_index('symbol')
        .groupby([pd.Grouper(freq='M'), 'symbol'])
        .last()
    )

    # Price-based features: monthly return lags + RSI-like features
    price_feats = {}
    for ticker in universe:
        if ticker not in prices.columns:
            continue
        px = prices[ticker].dropna()
        monthly_px = px.resample('M').last()

        ret_1m = monthly_px.pct_change(1)
        ret_3m = monthly_px.pct_change(3)
        ret_6m = monthly_px.pct_change(6)
        vol_1m = np.log(px).diff().resample('M').std() * np.sqrt(21)

        # Sentiment-price divergence: sent 1M change vs price 1M change
        for dt_idx in monthly_px.index:
            price_feats.setdefault(dt_idx, {})[ticker] = {
                'ret_1m': ret_1m.get(dt_idx, np.nan),
                'ret_3m': ret_3m.get(dt_idx, np.nan),
                'ret_6m': ret_6m.get(dt_idx, np.nan),
                'vol_1m': vol_1m.get(dt_idx, np.nan),
            }

    rows = []
    for date, tickers_data in price_feats.items():
        for ticker, vals in tickers_data.items():
            vals['date']   = date
            vals['symbol'] = ticker
            rows.append(vals)

    price_df = pd.DataFrame(rows).set_index(['date', 'symbol'])

    # Merge sentiment + price
    combined = monthly_sent.join(price_df, how='inner')

    # Add sentiment-price divergence: sent momentum - price momentum (standardised)
    combined['spd'] = (
        combined.groupby(level='date')['sent_momentum_3_10'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
        - combined.groupby(level='date')['ret_1m'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
    )

    return combined


ml_features = build_ml_features(features_df, prices, UNIVERSE)
print(f'ML feature matrix: {ml_features.shape}')
ml_features.head()

In [ ]:
# ── Walk-Forward ML Backtest ──────────────────────────────────────────
ML_FEATURE_COLS = ['sent_momentum_3_10', 'vol_weighted_sent', 'bull_bear_div_10d',
                   'sent_zscore', 'engagement_sent', 'influencer_signal',
                   'ret_1m', 'ret_3m', 'ret_6m', 'vol_1m', 'spd']

# Merge with forward returns
ml_with_rets = ml_features.join(fwd_long, how='inner').dropna(subset=ML_FEATURE_COLS + ['fwd_return_1m'])

dates_sorted = ml_with_rets.index.get_level_values('date').unique().sort_values()
TRAIN_WINDOW = 24   # months

ml_predictions = []

model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge',  Ridge(alpha=10.0))
])

for i in range(TRAIN_WINDOW, len(dates_sorted) - 1):
    train_dates = dates_sorted[i - TRAIN_WINDOW : i]
    test_date   = dates_sorted[i]

    train_mask = ml_with_rets.index.get_level_values('date').isin(train_dates)
    test_mask  = ml_with_rets.index.get_level_values('date') == test_date

    X_train = ml_with_rets.loc[train_mask, ML_FEATURE_COLS]
    y_train = ml_with_rets.loc[train_mask, 'fwd_return_1m']
    X_test  = ml_with_rets.loc[test_mask,  ML_FEATURE_COLS]

    if len(X_train) < 20 or len(X_test) == 0:
        continue

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    test_tickers = ml_with_rets.loc[test_mask].index.get_level_values('symbol')
    for ticker, pred in zip(test_tickers, preds):
        ml_predictions.append({'date': test_date, 'symbol': ticker, 'ml_pred': pred})

ml_pred_df = pd.DataFrame(ml_predictions).set_index(['date', 'symbol'])
print(f'ML predictions: {ml_pred_df.shape}')
ml_pred_df.head(10)

In [ ]:
# IC of ML model vs raw composite score
ml_eval = ml_with_rets[['fwd_return_1m']].join(ml_pred_df, how='inner').dropna()

ic_ml = compute_monthly_ic(ml_eval, signal_col='ml_pred', return_col='fwd_return_1m')
ic_raw = compute_monthly_ic(
    monthly_signal[['composite_score']].join(fwd_long, how='inner').dropna(),
    signal_col='composite_score'
)

# Align both series to same months for fair comparison
common_months = ic_ml.index.intersection(ic_raw.index)
ic_ml_aligned  = ic_ml.reindex(common_months)
ic_raw_aligned = ic_raw.reindex(common_months)

fig, ax = plt.subplots(figsize=(15, 5))
ic_raw_aligned.rolling(3).mean().plot(ax=ax, label='Raw Composite Score (3M rolling IC)', linewidth=1.8, color='steelblue')
ic_ml_aligned.rolling(3).mean().plot(ax=ax, label='ML-Enhanced Score (3M rolling IC)',    linewidth=1.8, color='mediumseagreen', linestyle='--')
ax.axhline(0, color='red', linestyle='--')
ax.set_title('IC Comparison: Raw Composite Score vs ML-Enhanced Score')
ax.set_ylabel('Spearman IC (3M rolling)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Raw Score   Mean IC: {ic_raw_aligned.mean():.4f}  |  ICIR: {ic_raw_aligned.mean() / ic_raw_aligned.std():.3f}')
print(f'ML-Enhanced Mean IC: {ic_ml_aligned.mean():.4f}  |  ICIR: {ic_ml_aligned.mean() / ic_ml_aligned.std():.3f}')

## 14. Strategy Summary & Key Takeaways

### How to Use Real Twitter Data

Replace `simulate_sentiment_data()` with one of these sources:

| Source | Cost | Fields |
|---|---|---|
| **StockTwits API** | Free | symbol, sentiment, message count |
| **Quiverquant** | ~$50/mo | tweet volume, sentiment per ticker |
| **Refinitiv** | Enterprise | high-quality, full-history |
| **Kaggle: Twitter Financial News** | Free | raw tweets, manual labelling needed |
| **VADER on raw tweets** | API cost | self-scored, full control |

### Strategy Extensions

1. **Sector Neutralisation** — rank within sectors to avoid sector timing bets
2. **Sentiment Volatility Filter** — avoid stocks with erratic sentiment (high variance = less reliable signal)
3. **Event-Driven Overlay** — spike detection around earnings to go flat
4. **Options Market Sentiment** — combine with put/call ratio or IV skew
5. **LLM-Based Sentiment Scoring** — replace VADER with Claude/GPT for nuanced financial NLP
6. **Portfolio Optimisation** — replace equal weights with mean-variance or risk parity
7. **Real-Time Signal** — move from monthly to daily rebalancing with intraday execution

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────
print('\n' + '═' * 60)
print('       TWITTER SENTIMENT STRATEGY — FINAL SCORECARD')
print('═' * 60)
print(stats_df.to_string())
print('═' * 60)
print(f'Signal IC Mean       : {ic_mean:.4f}')
print(f'Signal ICIR          : {icir:.3f}')
print(f'ML-Enhanced ICIR     : {(ic_ml_aligned.mean() / ic_ml_aligned.std()):.3f}')
print(f'L/S Spread (Q5-Q1)   : {(quintile_rets.iloc[-1] - quintile_rets.iloc[0])*100:.2f}% / month')
print('═' * 60)